# Middleware（中间件）简单来说就是Agent执行过程中的钩子函数，是LangChain1.x 的“王牌”能力
## 真实场景：
想根据问题复杂度动态 切换模型 ；

想 限制 某些用户只能调用部分工具；

想在工具报错时 自动重试 或返回兜底结果；

想在模型调用前 插入额外的系统提示 ；

想记录每一步的 执行日志 ，方便排查问题；

想在敏感信息出现时 阻断执行 ；

想在正式执行工具前增加 人工审批 。

工具链接：https://docs.langchain.com/oss/python/langchain/middleware/overview

# 1 常用中间件使用，LangChain提供了16个预置中间件，开箱即用

## 1.1 SummarizationMiddlewate中间件
作用：对历史消息列表进行 摘要&总结，达到压缩上下文，降低token消耗

原理：在达到触发条件时，大模型对历史消息进行摘要，将摘要结果作为HumanMessage，放到消息列表最开始的位置

## 举例1：测试trigger、keep参数
trigger：是一个列表，每个参数对应一个条件，当满足任意一个条件时除法摘要（token：token数量；message：历史消息数量；fraction：上下文比例，历史消息数量达到模型的max_input_tokens*fraction触发摘要

keep：摘要保存时的原始消息，支持3种（token：摘要时保存token数量；messags：摘要时保存历史消息；fraction：摘要时保留max_input_token*fraction个token）

In [44]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
import os

from dotenv import load_dotenv
from rich import print as rprint

# 加载配置文件，存在相同key采用当前覆盖
load_dotenv(override=True)

# 具体模型的key和url
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE   = os.getenv("DEEPSEEK_BASE_URL")
DEEPSEEK_MODEL_NAME   = os.getenv("DEEPSEEK_MODEL")


# 通义大模型
# 千问
API_KEY = os.getenv("DASHSCOPE_API_KEY")
BASE_URL   = os.getenv("DASHSCOPE_BASE_URL")
MODEL   = os.getenv("MODEL_NAME")

# 使用了fraction需要设置一个模型上下文大小，DeepSeek的profile为空，需要手动设置
custom_profile = {
    "max_input_tokens": 128_000
}


tongyi_model = init_chat_model(
    model="qwen-plus",
    profile=custom_profile,
    api_key = API_KEY,
    base_url = BASE_URL,
    model_provider="openai",

)



model = init_chat_model(
    model=DEEPSEEK_MODEL_NAME,
    model_provider="deepseek",
    profile=custom_profile,
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_API_BASE,
    extra_body={"thinking":{"type":"disabled"}}

)

messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]


# 创建agent
agent = create_agent(
    model=model,
    middleware=[
        SummarizationMiddleware(
            model=tongyi_model,
            trigger=[
                ("tokens",100),
                ("messages",5),
                ("fraction",0.001)
            ],
            # keep=("messages",3)
            keep=("tokens",3)
        )
    ]
)

res = agent.invoke({
    "messages": messages,
})

for check in res["messages"]:
    check.pretty_print()


================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT  
Establish initial rapport and identify participants in a casual, friendly introduction.

## SUMMARY  
The session began with a system instruction establishing the AI’s friendly demeanor. The user (self-identified as “老王”) initiated a greeting and asked for the AI’s identity. The AI introduced itself as “小王”, and the user acknowledged the introduction positively. The AI responded with playful, lighthearted humor (“你高兴得太早了”), signaling a tone of gentle teasing rather than literal concern — no substantive task, decision, or strategy was pursued beyond this introductory exchange. No options were rejected; the interaction remains purely social and exploratory.

## ARTIFACTS  
None

## NEXT STEPS  
Await the user’s next message to determine the actual task or goal — no concrete objective has yet been stated or initiated.
=======================

## 举例2：测试summary_prompt参数
该提示词包含{message}占位符，使得历史消息列表可以背插入


In [15]:
agent01 = create_agent(
    model=model,
    middleware=[
        SummarizationMiddleware(
            model=tongyi_model,
            trigger=[
                ("tokens",100),
                ("messages",6),
                ("fraction",0.001)
            ],
            keep=("messages",2),
            summary_prompt="对历史消息摘要如下\n{messages}"
        )
    ]
)

res01 = agent01.invoke({
    "messages": messages,
})

for check in res01["messages"]:
    check.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

历史消息摘要：  
- 系统设定：AI助手性格友好。  
- 老王自我介绍并询问AI身份；AI（自称“小王”）友好回应并报上名字。  
- 老王表示很高兴认识小王，双方完成初步友好问候与相识。
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message ==================================

（笑着摆摆手）别误会别误会，我就是开个玩笑。我的意思是，咱们这算正式认识了，以后可以多聊聊天嘛。你说是不是？


## 1.2 HumanInTheLoopMiddleware中间件
作用：在工具调用前中断Agent运行，等待用户对工具进行决策，可选决策有：approve(同意）、edit(编辑调用修改参数）、reject（拒绝）

参数：interrupt_on 工具名和中断策略，策略可以时True、False、InterruptOConfig

### 举例1 调用前中断

In [26]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage


from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
import os
from langchain.agents.middleware.human_in_the_loop import HumanInTheLoopMiddleware

from dotenv import load_dotenv
from rich import print as rprint

# 加载配置文件，存在相同key采用当前覆盖
load_dotenv(override=True)

# 具体模型的key和url
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE   = os.getenv("DEEPSEEK_BASE_URL")
DEEPSEEK_MODEL_NAME   = os.getenv("DEEPSEEK_MODEL")


@tool
def get_weather(city: str, is_forcast: bool = False) -> str:
    """
    查询指定城市天气

    Args:
    city: 城市名称
    is_forcast: 是否包含明日天气预报？
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天下雨"
    return res


@tool
def get_news() -> str:
    """
    查询当日新闻
    """
    return "中方三艘油轮通过霍尔木兹海峡"

@tool
def read_email_tool(email_id: str) -> str:
    """通过邮件ID读取内容的伪函数"""
    return f"邮件ID：{email_id}\n是空的"

@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """发送邮件伪函数"""
    print(">>> 真的执行发送邮件工具了")
    return f"发送给 {recipient} 的邮件标题是：{subject}，内容：{body}"

# 通义大模型
# 千问
API_KEY = os.getenv("DASHSCOPE_API_KEY")
BASE_URL   = os.getenv("DASHSCOPE_BASE_URL")
MODEL   = os.getenv("MODEL_NAME")

# 使用了fraction需要设置一个模型上下文大小，DeepSeek的profile为空，需要手动设置
custom_profile = {
    "max_input_tokens": 128_000
}


tongyi_model = init_chat_model(
    model="qwen-plus",
    profile=custom_profile,
    api_key = API_KEY,
    base_url = BASE_URL,
    model_provider="openai",

)



model = init_chat_model(
    model=DEEPSEEK_MODEL_NAME,
    model_provider="deepseek",
    profile=custom_profile,
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_API_BASE,
    extra_body={"thinking":{"type":"disabled"}}

)

messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]


# 创建agent
agent = create_agent(
    model=model,
    tools=[get_weather, get_news, read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "get_weather":True,
                "get_news":True,
                "read_email_tool":False,
                "send_email_tool":{
                    "allowed_decisions": ["approve", "reject"],
                    "description": "发送邮件中断啦"
                },
            },
            description_prefix ="工具中断中！！！"

        )
    ]
)

# 保证在一个会话中
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
{
"messages": [
HumanMessage(content="请帮我查询今天北京的天气"
    "查询今日新闻"
    "查看ID为 'sk2131421' 的邮件内容，"
    "向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'"
    "同时做这四件事")
]
},
config=config,
)

print("==== 第一次 invoke 返回 ====")
print("========= 原始响应 =========")
rprint(response)
print("========= 美化输出 =========")
for msg in response["messages"]:
    msg.pretty_print()
# 关键：看中断信息
interrupts = response.get("__interrupt__", [])
print("========== interrupts ==========")
rprint(interrupts)
# print("==== 逐个打印 interrupt 请求 ====")
action_requests = interrupts[0].value["action_requests"]
for action_request in action_requests:
    rprint(action_request)


==== 第一次 invoke 返回 ====
========= 原始响应 =========


{
    'messages': [
        HumanMessage(
            content="请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 
的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事",
            additional_kwargs={},
            response_metadata={},
            id='816a54c4-a0cc-4af6-b903-ff88996cf32b'
        ),
        AIMessage(
            content='我来同时为您执行这四件事：查询北京天气、获取今日新闻、读取邮件、发送邮件。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 188,
                    'prompt_tokens': 515,
                    'total_tokens': 703,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 512
                    },
                    'prompt_cache_hit_tokens': 512,
                    'prompt_cache_miss_tokens': 3
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '71e298e5-e1d3-486b-bad6-fe4d137076d3',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019fff8d-a817-70c2-ac2d-1740e44c993b-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京'},
                    'id': 'call_00_VsyTWFvUNVIZnBuICDtt4673',
                    'type': 'tool_call'
                },
                {'name': 'get_news', 'args': {}, 'id': 'call_01_NtX2VE2lcA0AyMrHkPfu1725', 'type': 'tool_call'},
                {
                    'name': 'read_email_tool',
                    'args': {'email_id': 'sk2131421'},
                    'id': 'call_02_fX8fMP2Y0wV00tzYOw0R6281',
                    'type': 'tool_call'
                },
                {
                    'name': 'send_email_tool',
                    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                    'id': 'call_03_EQIwgCcdW3LN3XG1Oxtd2104',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 515,
                'output_tokens': 188,
                'total_tokens': 703,
                'input_token_details': {'cache_read': 512},
                'output_token_details': {}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'get_weather',
                        'args': {'city': '北京'},
                        'description': "工具中断中！！！\n\nTool: get_weather\nArgs: {'city': '北京'}"
                    },
                    {
                        'name': 'get_news',
                        'args': {},
                        'description': '工具中断中！！！\n\nTool: get_news\nArgs: {}'
                    },
                    {
                        'name': 'send_email_tool',
                        'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                        'description': '发送邮件中断啦'
                    }
                ],
                'review_configs': [
                    {'action_name': 'get_weather', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']},
                    {'action_name': 'get_news', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']},
                    {'action_name': 'send_email_tool', 'allowed_decisions': ['approve', 'reject']}
                ]
            },
            id='06c061fff716d53e061ac97430c578b1'
        )
    ]
}

========= 美化输出 =========
================================ Human Message =================================

请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事
================================== Ai Message ==================================

我来同时为您执行这四件事：查询北京天气、获取今日新闻、读取邮件、发送邮件。
Tool Calls:
  get_weather (call_00_VsyTWFvUNVIZnBuICDtt4673)
 Call ID: call_00_VsyTWFvUNVIZnBuICDtt4673
  Args:
    city: 北京
  get_news (call_01_NtX2VE2lcA0AyMrHkPfu1725)
 Call ID: call_01_NtX2VE2lcA0AyMrHkPfu1725
  Args:
  read_email_tool (call_02_fX8fMP2Y0wV00tzYOw0R6281)
 Call ID: call_02_fX8fMP2Y0wV00tzYOw0R6281
  Args:
    email_id: sk2131421
  send_email_tool (call_03_EQIwgCcdW3LN3XG1Oxtd2104)
 Call ID: call_03_EQIwgCcdW3LN3XG1Oxtd2104
  Args:
    recipient: 15641685664@qq.com
    subject: 哈哈哈
    body: 你好啊
========== interrupts ==========


[
    Interrupt(
        value={
            'action_requests': [
                {
                    'name': 'get_weather',
                    'args': {'city': '北京'},
                    'description': "工具中断中！！！\n\nTool: get_weather\nArgs: {'city': '北京'}"
                },
                {'name': 'get_news', 'args': {}, 'description': '工具中断中！！！\n\nTool: get_news\nArgs: {}'},
                {
                    'name': 'send_email_tool',
                    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                    'description': '发送邮件中断啦'
                }
            ],
            'review_configs': [
                {'action_name': 'get_weather', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']},
                {'action_name': 'get_news', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']},
                {'action_name': 'send_email_tool', 'allowed_decisions': ['approve', 'reject']}
            ]
        },
        id='06c061fff716d53e061ac97430c578b1'
    )
]

{
    'name': 'get_weather',
    'args': {'city': '北京'},
    'description': "工具中断中！！！\n\nTool: get_weather\nArgs: {'city': '北京'}"
}

{'name': 'get_news', 'args': {}, 'description': '工具中断中！！！\n\nTool: get_news\nArgs: {}'}

{
    'name': 'send_email_tool',
    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
    'description': '发送邮件中断啦'
}

### 举例2 中断决策

In [27]:
from langgraph.types import Command

# 如果有中断，说明进入人在环了
weather_decision = {
    "type": "edit",
    "edited_action": {
        "name": "get_weather",
        "args": {"city": "杭州市", "is_forcast": True}
}
}
news_decision = {
    "type": "approve",
}
send_email_decision = {
    "type": "approve"
}
decisions = {
    "decisions": []
}
# 决策的顺序必须和返回的中断请求顺序一致
for action_request in action_requests:
    if action_request["name"] == "get_weather":
        decisions["decisions"].append(weather_decision)
    if action_request["name"] == "get_news":
        decisions["decisions"].append(news_decision)
    if action_request["name"] == "send_email_tool":
        decisions["decisions"].append(send_email_decision)

if interrupts:
# 审批通过
    resumed_response = agent.invoke(
        Command(resume=decisions),
        config=config, # 必须是同一个 thread_id
    )

print("==== 审批后继续执行 ====")
for msg in resumed_response["messages"]:
    msg.pretty_print()

>>> 真的执行发送邮件工具了
==== 审批后继续执行 ====
================================ Human Message =================================

请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事
================================== Ai Message ==================================

我来同时为您执行这四件事：查询北京天气、获取今日新闻、读取邮件、发送邮件。
Tool Calls:
  get_weather (call_00_VsyTWFvUNVIZnBuICDtt4673)
 Call ID: call_00_VsyTWFvUNVIZnBuICDtt4673
  Args:
    city: 杭州市
    is_forcast: True
  get_news (call_01_NtX2VE2lcA0AyMrHkPfu1725)
 Call ID: call_01_NtX2VE2lcA0AyMrHkPfu1725
  Args:
  read_email_tool (call_02_fX8fMP2Y0wV00tzYOw0R6281)
 Call ID: call_02_fX8fMP2Y0wV00tzYOw0R6281
  Args:
    email_id: sk2131421
  send_email_tool (call_03_EQIwgCcdW3LN3XG1Oxtd2104)
 Call ID: call_03_EQIwgCcdW3LN3XG1Oxtd2104
  Args:
    recipient: 15641685664@qq.com
    subject: 哈哈哈
    body: 你好啊
================================= Tool Message =================================
Name: get_weather

杭州市今天天气不错
明天下雨
================

## 1.3 PIIMiddleware中间件
敏感信息保护，用于检查和处理对话中的个人身份信息（Personally Identifiable Information，PII），支持自定义处理决策

参数：

pii_type:email、credit_card（信用卡号）、URL（网址）、Mac_address(设备Mac地址）、IP（IP地址）

strategy处理策略：redact（将敏感信息字符串代替）、mask（***替换）、hash（哈希值替换）、block（直接抛异常）

detector：自定义检测PII函数或者正则表达式

applay_to_input:是否在调用前检测，一般都是采用这种模式

applay_to_ooutput:是否在模型调用后检测

apply_to_tool_results 是否在工具调用后检测其输出 默认False

### 举例1

In [29]:
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        PIIMiddleware("url", strategy="hash", apply_to_input=True),
        PIIMiddleware("mac_address", strategy="mask", apply_to_input=True),
        PIIMiddleware("ip", strategy="block", apply_to_input=True),
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("""
    帮我向 156168188@qq.com 发送一封邮件
    同时查看银行卡号： 5105-1051-0510-5100 的余额
    访问 https://localhost:12345
    确认这是不是 MAC地址： 11-11-11-11-11-11
    """)]
})
# for msg in response["messages"]:
#     msg.pretty_print()

try:
    response1 = agent.invoke({
        "messages": [HumanMessage("看看这个 IP 能不能 ping 通：192.168.10.1")]
    })
except Exception as e:
    print('=' * 30, '-> 抛异常 <-', '=' * 30)
    print(f"检测到IP，抛出异常：{e}")

============================== -> 抛异常 <- ==============================
检测到IP，抛出异常：Detected 1 instance(s) of ip in text content


### 举例2

In [33]:
import re
# 自定义检测函数
def detect_phone_number(content: str):
    return [
    {
    "text": m.group(0), # 提取出具体匹配到的 11 位数字文本（例如"13800138000"）
    "start": m.start(), # 这段数字在原文本中的“起始索引位置”（从 0 开始算）
    "end": m.end() # 这段数字在原文本中的“结束索引位置”
    } for m in re.finditer(r"[0-9]{11}", content)
    ]

# finditer是python正则中非常高效的一个方法，返回一个Iterator迭代器

text = "尚硅谷的电话是13812345678，康师傅的电话是13987654321。"
result = detect_phone_number(text)
print(result)


agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("api_key", strategy="hash", apply_to_input=True,
        detector=r"sk-[a-zA-Z0-9]+"),
        PIIMiddleware("phone_number", strategy="mask", apply_to_input=True,
        detector=detect_phone_number)
    ]
)
response = agent.invoke({
    "messages": [HumanMessage("""
    这是不是有效的 API_KEY： sk-awef23AFEfaafaefa
    帮我给这个号码打电话： 12345612345
    访问 https://localhost:12345
                              """)]
})
for msg in response["messages"]:
    msg.pretty_print()

[{'text': '13812345678', 'start': 7, 'end': 18}, {'text': '13987654321', 'start': 26, 'end': 37}]
================================ Human Message =================================


    这是不是有效的 API_KEY： <api_key_hash:6c678cc0>
    帮我给这个号码打电话： ****2345
    访问 https://localhost:12345
                              
================================== Ai Message ==================================

我无法确认这个 API 密钥是否有效，也无法帮你打电话或访问本地地址。API 密钥是敏感信息，它的有效性只能通过你实际使用的服务端验证，无法从哈希值直接判断。

如果你需要验证 API 密钥，可以通过以下方式自行确认：
1. 登录你创建该密钥的服务商控制台查看。
2. 在代码中尝试调用该 API 的测试接口，看返回结果是否成功。

至于电话号码和本地地址（localhost），请确保你已获得对方明确授权，且访问本地服务需在你自己设备上操作。安全起见，不要向陌生人透露你的电话号码或允许他人远程访问你的设备。


## 1.4 TodoListMiddleware中间件
赋予了Agent任务规划和追踪进度能力，可以应对复杂的多任务，比如一个大任务拆分成3个以上子任务，且前面步骤是后面步骤的前提时，如果不列todo，大模型可能容易忘记掉之前的最初目标

典型场景：

任务链路长、步骤多，且有严格的先后依赖关系

需要在前端 UI 界面实时展示 Agent 的“思考与执行进度”


参数说明：

system_prmpt:自定义指导todo提示词

tool_description:自定义write_tool工具的描述信息


### 举例1
1. pytest会扫描目录下所有以 test_ 开头或以 _test 结尾的文件，视为测试文件

2. 然后执行测试文件中所有以 test 开头的函数

3. 执行出错会打印到控制台，如上所示

4. 测试函数的逻辑是调用my_add.py中的add函数，得不到符合预期的结果则抛出异常。

In [39]:
from langchain.tools import tool
from pathlib import Path
import subprocess
WORKSPACE = Path("../middleware/todo_workspace")
@tool
def list_files(path: str = ".") -> str:
    """
    列出工作区指定目录下的文件和子目录。path 只能是相对路径。

    Args:
    path: 工作区下的相对路径，一定指向目录，默认为.，表示工作区根路径，不能访问工作区
    外的目录
    """
    target = (WORKSPACE / path).resolve()
    workspace_root = WORKSPACE.resolve()
    if not str(target).startswith(str(workspace_root)):
        return "错误：只允许访问工作区内的目录。"
    if not target.exists():
        return f"错误：目录不存在: {path}"
    if not target.is_dir():
        return f"错误：不是目录: {path}"
    items = sorted(target.iterdir(), key=lambda p: (p.is_file(),
    p.name.lower()))
    if not items:
        return f"目录为空: {path}"
    lines = []
    for item in items:
        rel = item.relative_to(workspace_root)
        kind = "[DIR]" if item.is_dir() else "[FILE]"
        lines.append(f"{kind} {rel.as_posix()}")
    return "\n".join(lines)
@tool
def read_file(path: str) -> str:
    """
    读取工作区中的文本文件内容。path 只能是相对路径。

    Args:
    path: 工作区内的文件名
    """
    file_path = (WORKSPACE / path).resolve()
    if not str(file_path).startswith(str(WORKSPACE.resolve())):
        return "错误：只允许读取工作区内的文件。"
    if not file_path.exists():
        return f"错误：文件不存在: {path}"
    return file_path.read_text(encoding="utf-8")
@tool
def write_file(path: str, content: str) -> str:
    """
    写入工作区中的文本文件。path 只能是相对路径。

    Args:
    path: 工作区内的文件名
    content: 写入文件的内容
    """
    file_path = (WORKSPACE / path).resolve()
    if not str(file_path).startswith(str(WORKSPACE.resolve())):
        return "错误：只允许写入工作区内的文件。"
    file_path.write_text(content, encoding="utf-8")
    return f"已写入文件: {path}"

@tool
def run_tests() -> str:
    """
    在工作区运行 pytest -q，并返回输出。
    不接收任何参数，返回格式为
    returncode=0|1
    STDOUT:
    STDERR:
    """
    try:
        result = subprocess.run(
        ["pytest", "-q"],
        cwd=str(WORKSPACE),
        capture_output=True,
        text=True,
        timeout=20,
        )
        return (
        f"returncode={result.returncode}\n\n"
        f"STDOUT:\n{result.stdout}\n\n"
        f"STDERR:\n{result.stderr}"
        )
    except Exception as e:
        return f"运行测试失败: {e}"

In [42]:
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain.messages import HumanMessage
from rich import print as rprint
# 1. 初始化 Agent
agent = create_agent(
    model=model,
    # write_todos 等工具，TodoListMiddleware 需要配合这些工具使用
    tools=[list_files, read_file, write_file, run_tests],
    # 引入 Todo 列表中间件
    middleware=[TodoListMiddleware()],
    system_prompt=(
    "你是一个代码修复助手。遇到多步骤任务时，先使用 write_todos 制定待办事项；"
    "然后读取文件、修复代码并运行测试。工作全部在工作区下进行。"
    ),
)
# 2. 使用invoke进行同步调用
print("正在执行 Agent 任务...")
final_state = agent.invoke(
{
    "messages": [
    HumanMessage(content="请测试并修复工作区下 my_add.py 文件中的代码")
    ]
}
)

# rprint(final_state)
# 3. 直观展示中间件产生的数据结果
print("\n" + "="*20 + " 1. 中间件拦截到的 TODO 列表 " + "="*20)

# TodoListMiddleware 运行期间，会自动将规划好的步骤注入到 state 的 "todos" 字段中
todos = final_state.get("todos", [])

if todos:
    for i, item in enumerate(todos, 1):
        # 兼容中间件可能返回的不同字典结构
        content = item.get("content") or item.get("task") or item.get("text") or str(item)
        status = item.get("status", "unknown")
        print(f"{i}. [{status}] {content}")
else:
    print("未检测到待办事项（可能 Agent 认为不需要规划，或未触发 write_todos 工具）")


print("\n" + "="*20 + " 2. Agent 最终修复回复 " + "="*20)
# 获取对话历史中的最后一条消息（即 Agent 的最终总结）
if final_state.get("messages"):
    print(final_state["messages"][-1].content)

正在执行 Agent 任务...

==================== 1. 中间件拦截到的 TODO 列表 ====================
未检测到待办事项（可能 Agent 认为不需要规划，或未触发 write_todos 工具）

==================== 2. Agent 最终修复回复 ====================
测试已全部通过！

## 问题总结

**问题原因**：`my_add.py` 中 `add` 函数的实现有误，将 `a + b` 错误地写成了 `a - b`，导致返回的是两个数的差值而非和值。例如 `add(2, 3)` 返回了 `-1` 而不是预期的 `5`。

**修复内容**：将函数体从 `return a - b` 改为 `return a + b`。

**验证结果**：所有测试用例（共 1 个测试函数，4 个断言）均通过，覆盖了正数、负数、零等场景。
